# Brent · `url_planning_app` (web: "URL planning application")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `url_planning_app` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "url_planning_app"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "url_planning_app"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["url_planning_app"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,4038,10.93
empty or blank,0,0.00
filled,32914,89.07
total,36952,100.00


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 4,038 | 10.93% |
| empty or blank | 0 | 0% |
| filled | 32,914 | 89.07% |

- **4,038 have no URL** (10.9%), close to the 4,025 with no `decision_target_date`; the
  overlap is not checked yet.

## Step 2 · Do the values match the schema type?

Count the Python type of every value in `df`: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [4]:
types = df["url_planning_app"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["url_planning_app"].value_counts().head(30).to_frame("count")

,count
url_planning_app,
str,32914
NoneType,4038


values that are not strings: 0


,count
url_planning_app,


### Findings · Step 2

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 32,914 |
| null | 4,038 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says (`string`). No type mismatch.

## Step 3 · Overlap with the records that have no `decision_target_date`

Count, with `_count` and the Brent filter, the records with no URL that also have no
target date, and split the records with no URL into FME Legacy records, empty records
(dropped at load) and the rest.

In [5]:
def count(filters: list[dict], not_filters: list[dict] | None = None) -> int:
    body = {"query": {"bool": {"filter": [brent_filter(), *filters], "must_not": not_filters or []}}}
    response = httpx.post(f"{BASE_URL}/applications/_count", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()["count"]


def missing(field: str) -> dict:
    return {"bool": {"must_not": [{"exists": {"field": field}}]}}


fme_legacy = {"term": {"bo_system.raw": "FME Legacy Data Import"}}
empty_record = {"bool": {"filter": [missing("status"), missing("decision"), missing("decision_date"), missing("description")]}}
no_url = missing("url_planning_app")
no_target = missing("decision_target_date")

display(pd.Series({
    "no url_planning_app": count([no_url]),
    "  also no decision_target_date": count([no_url, no_target]),
    "  has a decision_target_date": count([no_url], [no_target]),
    "no decision_target_date but has a URL": count([no_target], [no_url]),
}).to_frame("count"))

pd.Series({
    "no url_planning_app": count([no_url]),
    "  FME Legacy": count([no_url, fme_legacy]),
    "  empty records (dropped)": count([no_url, empty_record]),
    "  neither": count([no_url], [fme_legacy, empty_record]),
}).to_frame("count")

,count
no url_planning_app,4038
also no decision_target_date,4023
has a decision_target_date,15
no decision_target_date but has a URL,2


,count
no url_planning_app,4038
FME Legacy,3531
empty records (dropped),407
neither,100


### Findings · Step 3

Live API, 2026-11-10:

| | Count |
|---|---|
| no URL | 4,038 |
| also no `decision_target_date` | 4,023 |
| has a `decision_target_date` | 15 |
| no `decision_target_date` but has a URL | 2 |

| No URL | Count |
|---|---|
| FME Legacy records | 3,531 |
| empty records (dropped at load) | 407 |
| neither (real gaps) | 100 |

- **Almost the same records as the missing target dates:** 4,023 of the 4,038 have neither.
- **All FME Legacy records and all empty records have no URL**, the same pattern as
  `valid_date` and `decision_target_date`.
- **After cleaning, about 100 real records have no URL** (one of them, `OPDC-19_2236`
  from Portal, is an OPDC duplicate dropped at load).

## Step 4 · Legacy records and missing dates / URLs after cleaning

The same check for `valid_date`, `decision_target_date` and `url_planning_app`, counted on
the records that remain after cleaning (empty records, the 3 test records and the 4 OPDC
duplicates left out). The numbers used in the semantic YAMLs.

In [6]:
dropped = {"ids": {"values": [
    "Brent-55_0003", "Brent-55_0015", "Brent-55_0100",
    "OPDC-18_1022", "OPDC-18_2581", "OPDC-19_2236", "OPDC-19_2890",
]}}
after_cleaning = [empty_record, dropped]

rows = {"records after cleaning": count([], after_cleaning), "legacy records": count([fme_legacy], after_cleaning)}
table = pd.DataFrame({
    field: {
        "missing": count([missing(field)], after_cleaning),
        "  legacy records": count([missing(field), fme_legacy], after_cleaning),
        "  other (real gaps)": count([missing(field)], [fme_legacy, *after_cleaning]),
        "legacy records that have one": count([fme_legacy], [missing(field), *after_cleaning]),
    }
    for field in ["valid_date", "decision_target_date", "url_planning_app"]
})
display(pd.Series(rows).to_frame("count"))
table

,count
records after cleaning,36538
legacy records,3528


,valid_date,decision_target_date,url_planning_app
missing,3602,3614,3627
legacy records,3528,3528,3528
other (real gaps),74,86,99
legacy records that have one,0,0,0


### Findings · Step 4

Live API, 2026-11-10, 36,538 records after cleaning, 3,528 of them legacy records (3 of
the 3,531 were OPDC duplicates):

| | `valid_date` | `decision_target_date` | `url_planning_app` |
|---|---|---|---|
| missing | 3,602 | 3,614 | 3,627 |
| legacy records | 3,528 | 3,528 | 3,528 |
| other (real gaps) | 74 | 86 | 99 |
| legacy records that have one | 0 | 0 | 0 |

- **A legacy record never has a valid date, a target date or a URL.** Every other missing
  value is a real gap: 74, 86 and 99 records.